# Knowledge base (ChromaDB)

In [273]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch
import pandas as pd
import chromadb
from tqdm.notebook import tqdm

CSV_PATH = "./knowledge/clean_youtube_watch_log.csv"
THUMBNAIL_PATH = "./knowledge/thumbnails"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection(name="youtube_videos")


def embed_text(text: str):
    """Embed a text using CLIP"""
    inputs = proc(
        text=[text], return_tensors="pt", truncation=True, padding=True
    )  # WARNING! Truncation may lead to loss of information for long texts
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True)  # Normalize

def embed_image(image: Image.Image):
    """Embed an image using CLIP"""
    inputs = proc(images=image, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True)  # Normalize


# Find new videos to eventually add to the collection
df = pd.read_csv(CSV_PATH)
ids_to_add = set(df["video_id"]) - set(collection.get(ids=None)["ids"])
print(f"Found {len(ids_to_add)} new videos to add to the collection.")

for video_id in tqdm(ids_to_add):
    video_data = df[df["video_id"] == video_id].iloc[0]
    user_id = video_data["user_id"]
    title = video_data["video_title"]
    thumbnail = Image.open(f"{THUMBNAIL_PATH}/{video_id}.jpg")

    # Embed text and image, then fuse them (weighted average)
    text_emb = embed_text(title)
    img_emb = embed_image(thumbnail)
    fused_emb = 0.7 * text_emb + 0.3 * img_emb

    # Add embeddings to the collection
    collection.add(
        ids=[video_id],             # identifier for each entry (e.g., video ID)
        embeddings=[fused_emb],     # list of embeddings (fused text + image)
        documents=[user_id],        # optional metadata (e.g., user ID or other info)
    )

Found 0 new videos to add to the collection.


0it [00:00, ?it/s]

# Building the Agent (LangChain)

In [274]:
from dataclasses import dataclass
from typing import Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from PIL import Image
from typing import Literal
from typing import Optional
from langgraph.types import Command
from langchain.agents import AgentState
from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain.messages import ToolMessage
from dataclasses import asdict
from langchain_groq import ChatGroq
from dataclasses import field
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
import random

random.seed(42)

with open('llm-api-key.txt') as f:
    api_key = f.readline().strip()

# llm = ChatOllama(model="qwen3.5:2B", temperature=0)   # or qwen3:0.6b-q4_K_M
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key,
    temperature=0.4,
)
# llm = ChatGoogleGenerativeAI(
#     model="gemini-2.5-flash-lite",
#     temperature=0,
#     api_key=api_key,
#     convert_system_message_to_human=True,
# )

## 1. Data types

In [275]:
@dataclass
class WatchItem:
    user_id: int
    video_id: str
    video_title: str
    timestamp: str      # ISO format

    @property
    def thumbnail(self) -> Image.Image:
        return Image.open(f'./knowledge/thumbnails/{self.video_id}.jpg')
    
    @classmethod
    def from_row(cls, row) -> "WatchItem":
        return cls(
            user_id=row['user_id'],
            video_id=row['video_id'],
            video_title=row['video_title'],
            timestamp=row['watch_date'].isoformat()
        )

@dataclass
class BiasProfile:
    emotional_tone: Literal["low", "medium", "high"]
    sensationalism: Literal["low", "medium", "high"]
    topical_narrowing: Literal["low", "medium", "high"]
    echo_chamber_effect: Literal["low", "medium", "high"]
    dominant_topics: list[str]
    evidence_titles: list[str]
    overall_profile: str = field(
        metadata={"description": "Some comments from the model explaining why the scores were assigned."}
    )
    confidence: float = field(
        default=0.0,
        metadata={"description": "A confidence score (0-1) indicating how confident the model is in its bias assessment."}
    )

## 2. Tools

In [ ]:
@tool
def retrieve_session(user_id: int, sort_asc: bool, limit: int, runtime: ToolRuntime[None, AgentState]) -> Command:
    """Loads a user's watch session.
    Args:
        user_id (int): user ID
        sort_asc (bool): sort oldest first if True
        limit (int): max results
    Returns:
        list[WatchItem]: user's watch session
    """
    df = pd.read_csv(CSV_PATH)
    df = df[df['user_id'] == user_id]   # Filter by user ID
    df["watch_date"] = pd.to_datetime(df["watch_date"])
    df = df.sort_values("watch_date", ascending=sort_asc)
    df = df.head(limit) if limit else df
    result = df.apply(WatchItem.from_row, axis=1).tolist()  # Convert to WatchItem list
    return Command(
        update ={
            "watch_history": result,    # Update agent state with the retrieved watch history
            "messages": [ToolMessage(
                content="Retrieved watch history: " + str([asdict(r) for r in result]),
                tool_call_id=runtime.tool_call_id
            )],
        },
    )


@tool
def analyze_bias_profile(watch_history: list[WatchItem], runtime: ToolRuntime[None, AgentState]) -> Command:
    """ Runs a bias analysis on the provided watch history using a sub-agent.
    Args:
        watch_history (list[WatchItem]): The user's watch session.
    Returns:
        Command: Updates the agent state with the bias profile.
    """
    global bias_analyzer_agent
    result = bias_analyzer_agent.invoke(HumanMessage(content="Analyze this watch history for bias: " + str(watch_history)))
    result = result["structured_response"]    # BiasProfile
    return Command(
        update = { 
            "bias_profile": result,
            "messages": [ToolMessage(
                content="Analyzed watch history bias: " + str(result),
                tool_call_id=runtime.tool_call_id
            )],
        },
    )

@tool
def retrieve_thumbnail(video_id: list[str]) -> list[Image.Image]:
    """Retrieves the thumbnail image for one or more video IDs.
    Args:
        video_id (list[str]): The video ID for which to retrieve the thumbnail.
    Returns:
        list[Image.Image]: The thumbnail images associated with the video IDs.
    """
    return [Image.open(f"{THUMBNAIL_PATH}/{id}.jpg") for id in video_id]


@tool
def find_similar_videos(user_id: Optional[str], title: Optional[str], thumbnail: Optional[Image.Image], limit: int) -> list[WatchItem]:
    """Invokes the VectorDB to find similar videos by title and/or thumbnail. Can limit to videos watched by the user if necessary.
    Args:
        user_id (str, optional): If provided, limits the search to videos watched by this user.
        title (str, optional): The title of the video to find similarities for.
        thumbnail (Image.Image, optional): The thumbnail of the video to find similarities for.
        limit (int): The number of similar videos to return. Defaults to 5.
    Returns:
        list[WatchItem]: A list of similar videos.
    """
    # Embed the input title and thumbnail
    text_emb = embed_text(title) if title else None
    img_emb = embed_image(thumbnail) if thumbnail else None
    if text_emb is not None and img_emb is not None:
        fused_emb = 0.7 * text_emb + 0.3 * img_emb
    else:
        fused_emb = text_emb or img_emb  # Use whichever is available, or None if both are None

    # Query the vector database for similar videos
    similar_ids = collection.query(
        query_embeddings=fused_emb,
        n_results=limit or 5,
        where={"user_id": user_id} if user_id else None
    ).get("ids", [[]])[0]  # Get the IDs

    return (
        df[df["video_id"].isin(similar_ids)]
        .apply(WatchItem.from_row, axis=1)
        .tolist()
    )


TOOLS = [retrieve_session, analyze_bias_profile, find_similar_videos]

## 3. Building the graph

In [ ]:
# Global state definition for the graph
class AgentState(AgentState):
    user_id: int
    watch_history: list[WatchItem]                          # @dataclass
    bias_profile: Optional[BiasProfile]
    messages: Annotated[list[BaseMessage], add_messages]    # automatically append 


# ---------- Node 1: General-Purpose LLM agent  ----------
SYSTEM_PROMPT = """
You are a YouTube watch-history analysis agent for rabbit hole and bias prevention.

Rules:
- Check the available tools, and invoke them as needed.
- Critically evaluate if more information is needed before responding to the user and invoke tools as needed.
- Be cautious and retrieve sufficient evidence before making any claims, by means of tool calls and watch history analysis.
- If asked to analyze bias, polarization, sensationalism, clickbait, emotional tone, or rabbit-hole effects, first ensure watch history is available then use the analyze bias_profile tool to get a structured bias profile.
- Provide evidence for any claims you make, referring to a few examples rooted in the data.
- Always check for state updates from tool calls and use that information in your reasoning.
- For factual questions, answer directly from the retrieved watch history.
- Be cautious and do not overclaim ideology.

Recommended structure (not mandatory):
Some of the videos you referred to are: 
- ...
- (...make 5 examples...)
- ...

Based on this watch history, (...explanation of bias profile...) 
"""

agent = create_agent(
    llm,
    tools=TOOLS,
    state_schema=AgentState,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(serde=JsonPlusSerializer(
        allowed_msgpack_modules=[
            ("__main__", "WatchItem"),
            ("__main__", "BiasProfile"),
        ]
    )),
)


# ---------- Node 2: Bias Analysis sub-agent  ----------
BIAS_AGENT_PROMPT = """
Analyze this YouTube watch session for possible rabbit-hole effects.

Return a structured bias profile.

Rules:
- Use only the provided records.
- Be cautious.
- Do not overclaim ideology.
- If evidence is weak, say unclear.
"""

bias_analyzer_agent = create_agent(
    llm,
    state_schema=AgentState,
    system_prompt=BIAS_AGENT_PROMPT,
    response_format=BiasProfile,
)

# 4. Testing the agent
> **⚠️ WARNING: Don't forget to run cell above!**

In [278]:
user_id = "25"
thread_id = random.randint(1000, 9999)

print("Agent 🤖 is ready. Type 'exit' to quit.\n")
while True:
    user = input("\nYou: ").strip()
    if user == '' or user.lower() == "exit":
        break

    prompt = (f"The user_id is {user_id}. Answer to this question: {user}")
    stream = agent.stream(
        { "messages" : [HumanMessage(content=prompt)] },
        config={ "configurable": {"thread_id": thread_id} },    # add memory
        stream_mode="messages",
        version="v2",
    )

    print(f"\n\nYou 🙋‍♂️: {user}\n\nAgent 🪨: 🤔 ...")

    for part in stream:
        if part["data"][0].type != "AIMessageChunk" or not part["data"][0].content:
            continue
        print(part["data"][0].content, end="")

print("\n\nAgent 🪨: Okay then you don't like me. Goodbye.")

Agent 🤖 is ready. Type 'exit' to quit.



You 🙋‍♂️: what are the last 30 videos i have watched

Agent 🪨: 🤔 ...
The last 30 videos you have watched are:

- SkiVel
- Lords mobile! What are familiars, And how do you use them
- DILBAR Full Song | Satyameva Jayate | John Abraham Nora Fatehi | Tanishk B Neha Kakkar Ikka Dhvani
- Peekaboo Playground | Kids Songs | Super Simple Songs
- Goal # 5 | All Aboard For Global Goals! | Thomas & Friends
- Party Street | Hi-5 Season 17 Songs of the Week and more Kids Songs
- The Evil Jungle | Ben 10 | Cartoon Network
- Powerpuff Girls Was About a Wrestler?! | WHAT THEY GOT RIGHT
- StoryBots | Learn The Planets In The Solar System | Outer Space Songs For Kids | Netflix Jr
- 🔴 POCOYO in ENGLISH - Having a Ball 🔴 | Full Episodes | VIDEOS and CARTOONS FOR KIDS
- Tweens Make Assumptions About College Students | Reverse Assumptions
- NAME THE BABY! 24 weeks Bump Update + LIFE UPDATE!
- Learning to Speak Turkey | BBC Earth
- Simple Simon | Nursery Rhymes | Kids